In [ ]:
import sys
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from tabpfn import TabPFNRegressor

sys.path.append("..")
from src.data_generation.data_preperation import grid_from_cfg
from src.data_generation.noise import noisy_data_preparation, make_noisy_stratified_eval_set
from src.model.finetune import finetune
from src.model.SSVI import fit_ssvi, predict_ssvi
from src.evaluation.surface_eval import (
    eval_surfaces, quantile_coverage, inside_spread_fraction,
)

cfg = yaml.safe_load(open("../config.yaml"))
ttms, ks = grid_from_cfg(cfg)

warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
# overnight run: bid/ask quotes (X = [k, tau, side], side -1=bid/+1=ask/0=true),
# uniform 3-60 quote locations per surface (= 6-120 context rows), per-surface
# lognormal noise regime, clean full-grid targets.
# val: same 20 surfaces at {5, 10, 20, 40, 60} quotes every 5 epochs.
# NOTE: the "by n_ctx" labels in the log count context ROWS, i.e. 2x the quotes.
RUN_NAME = "ssvi_noisy_uniform_3_60"
data_provider = partial(noisy_data_preparation, cfg, n_context=(3, 60))
val_data = make_noisy_stratified_eval_set(cfg, n_surfaces=20, context_sizes=[5, 10, 20, 40, 60])

finetune(data_provider, run_name=RUN_NAME, n_epochs=300, n_surfaces_per_epoch=200,
         batch_size=4, val_data=val_data, val_every=5)

In [ ]:
N_ESTIMATORS = 1  # finetuned with 1

def load_finetuned(run_name, which="final"):
    model = TabPFNRegressor(
        fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS,
        inference_config={"FINGERPRINT_FEATURE": False},
    )
    model._initialize_model_variables()
    state = torch.load(f"../checkpoints/{run_name}/{which}.pt", map_location="cpu")
    model.model_.load_state_dict(state)
    return model, state

baseline = TabPFNRegressor(n_estimators=N_ESTIMATORS, inference_config={"FINGERPRINT_FEATURE": False})
noisy_ft, noisy_state = load_finetuned(RUN_NAME)
clean_ft, clean_state = load_finetuned("ssvi_uniform_context_3_30")

In [ ]:
def split_quotes(train):
    """(X_2feat, mid, half_spread) per surface from the bid/ask context rows."""
    out = []
    for X, y in train:
        n = len(y) // 2
        out.append((X[:n, :2], (y[:n] + y[n:]) / 2, (y[n:] - y[:n]) / 2))
    return out


def refit_metrics(train, test, weighted=False):
    """SSVI refit on mids (OLS, or WLS with observable 1/spread weights); MAE/MAPE vs truth."""
    maes, mapes = [], []
    for (X2, mid, s), (Xq, yq) in zip(split_quotes(train), test):
        w = 1 / np.maximum(2 * mid * X2[:, 1] * s, 1e-10) if weighted else None
        params, _ = fit_ssvi(X2, mid, cfg, weights=w)
        pred = predict_ssvi(params, ttms, ks).ravel()
        maes.append(np.mean(np.abs(pred - yq)))
        mapes.append(np.mean(np.abs((pred - yq) / yq)) * 100)
    return np.mean(maes), np.mean(mapes)

In [ ]:
# main sweep: quote count x noise regime, all models on the SAME test draws.
# mid-input models (baseline / clean-FT / refit) get (bid+ask)/2 with 2-feature X -
# the fair use for models that never saw the side convention.
N_CTX_VALUES = [3, 5, 8, 10, 15, 20, 40, 60]
REGIMES = [0, 0.5, 1, 2]
N_TEST = 25

for m in REGIMES:
    print(f"\n=== noise regime m={m} (MAE) ===")
    print(f"{'n_ctx':>6} {'noisy FT':>10} {'base(mid)':>10} {'cleanFT(mid)':>13} {'refit OLS':>10} {'refit WLS':>10}")
    for n_ctx in N_CTX_VALUES:
        tr, te = noisy_data_preparation(cfg, N_TEST, n_ctx, regime=m)
        te2 = [(X[:, :2], y) for X, y in te]
        mids = [(X2, mid) for X2, mid, _ in split_quotes(tr)]

        f = eval_surfaces(noisy_ft, tr, te, cfg, reload_state=noisy_state)
        b = eval_surfaces(baseline, mids, te2, cfg)
        c = eval_surfaces(clean_ft, mids, te2, cfg, reload_state=clean_state)
        r_o = refit_metrics(tr, te, weighted=False)
        r_w = refit_metrics(tr, te, weighted=True)
        print(f"{n_ctx:>6} {f[0]:>10.4f} {b[0]:>10.4f} {c[0]:>13.4f} {r_o[0]:>10.4f} {r_w[0]:>10.4f}")

In [ ]:
for m in [0.5, 1, 2]:
    for n_ctx in [5, 20, 60]:
        tr, te = noisy_data_preparation(cfg, 20, n_ctx, regime=m)
        cov = quantile_coverage(noisy_ft, tr, te, reload_state=noisy_state)
        print(f"m={m:<4} n_ctx={n_ctx:>3} | " + "  ".join(f"{lv:.0%} interval: {c:.2f}" for lv, c in cov.items()))

In [ ]:
for m in [0.5, 1, 2]:
    tr, _ = noisy_data_preparation(cfg, 20, 20, regime=m)
    frac = inside_spread_fraction(noisy_ft, tr, reload_state=noisy_state)
    print(f"m={m}: {frac:.1%} of predictions inside the quoted spread")

In [ ]:
N_QUOTES = 25
tr, te = noisy_data_preparation(cfg, 1, N_QUOTES, regime=2.0)
(Xn, yn), (Xq, yq) = tr[0], te[0]
noisy_ft.fit(Xn, yn)
noisy_ft.model_.load_state_dict(noisy_state)
pred = noisy_ft.predict(Xq)

n = len(yn) // 2
kq, tq, bid, ask = Xn[:n, 0], Xn[:n, 1], yn[:n], yn[n:]
fig, axes = plt.subplots(5, 3, figsize=(15, 18), sharex=True, sharey=True)
for t_i, ax in enumerate(axes.ravel()):
    sel = slice(t_i * len(ks), (t_i + 1) * len(ks))
    ax.plot(ks, yq[sel], "k-", label="true")
    ax.plot(ks, pred[sel], "C0--", label="prediction")
    q = np.isclose(tq, ttms[t_i])
    if q.any():
        ax.errorbar(kq[q], (bid[q] + ask[q]) / 2, yerr=(ask[q] - bid[q]) / 2,
                    fmt="C3.", capsize=3, label="quotes (bid/ask)")
    ax.set_title(f"tau={ttms[t_i]:.3f}  ({int(q.sum())} quotes)", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("k")
for ax in axes[:, 0]:
    ax.set_ylabel("IV")
axes[0, 0].legend()
plt.tight_layout()
plt.show()